In [0]:
dbutils.secrets.listScopes()

[]

In [0]:
jdbc_url = "jdbc:sqlserver://sql-frauddb-kp.database.windows.net:1433;database=frauddb;encrypt=true;trustServerCertificate=false;loginTimeout=30;"

df_txn = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", "adminkp")
    .option("password", "Rocky1234")
    .option("fetchsize", "10000")   # improves throughput for large reads
    .load()
)

In [0]:
print(df_txn.count())
df_txn.printSchema()
display(df_txn.limit(10))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5369054448094678>, line 1
----> 1 print(df_txn.count())
      2 df_txn.printSchema()
      3 display(df_txn.limit(10))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1929     query = self._plan.to_proto(self._session.client)
-> 1930     table, schema, self._execution_info = self._session.client.to_table(
   1931         query, self._plan.observations
   1

In [0]:
print(jdbc_url)

jdbc:sqlserver://sql-frauddb-kp.database.windows.net:1433;database=frauddb;encrypt=true;trustServerCertificate=false;loginTimeout=30;


In [0]:
jdbc_url = "jdbc:sqlserver://sql-frauddb-kp.database.windows.net:1433;database=frauddb;encrypt=true;trustServerCertificate=false;loginTimeout=30;"

df_txn = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", "adminkp")
    .option("password", "Rocky1234")
    .option("fetchsize", "10000")
    .load()
)

try:
    print(df_txn.count())
except Exception as e:
    print(str(e))

13305915


In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS fraud_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_project.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_project.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_project.gold")

DataFrame[]

In [0]:
(df_txn.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_project.bronze.transactions_data")
)

In [0]:
spark.sql("SELECT COUNT(*) FROM fraud_project.bronze.transactions_data").show()
display(spark.sql("SELECT * FROM fraud_project.bronze.transactions_data LIMIT 10"))

+--------+
|COUNT(*)|
+--------+
|13305915|
+--------+



id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00.000Z,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01T00:02:00.000Z,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01T00:02:00.000Z,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01T00:05:00.000Z,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01T00:06:00.000Z,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null
7475333,2010-01-01T00:07:00.000Z,1807,165,$4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,null
7475334,2010-01-01T00:09:00.000Z,1556,2972,$77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475335,2010-01-01T00:14:00.000Z,1684,2140,$26.46,Online Transaction,39021,ONLINE,null,null,4784,null
7475336,2010-01-01T00:21:00.000Z,335,5131,$261.58,Online Transaction,50292,ONLINE,null,null,7801,null
7475337,2010-01-01T00:21:00.000Z,351,1112,$10.74,Swipe Transaction,3864,Flushing,NY,11355.0,5813,null


In [0]:
df_cards = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.cards_data")
    .option("user", "adminkp")
    .option("password", "Rocky1234")
    .load()
)

print(df_cards.count())
display(df_cards.limit(10))

6146


id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,YES,2,$33900,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,YES,1,$11600,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,YES,1,$19948,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,YES,2,$16400,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,YES,2,$19439,01/1997,2007,No
5,619,Visa,Debit,4657824650820465,04/2024,245,YES,2,$21883,01/1997,2012,No
6,1046,Amex,Credit,394584924614148,02/1999,302,YES,2,$9400,01/1998,2011,No
7,511,Mastercard,Debit,5585238056278288,03/2005,749,YES,1,$9664,01/1998,2011,No
8,1107,Mastercard,Credit,5462760953855576,09/2021,665,NO,2,$10300,01/1998,2006,No
9,1046,Amex,Credit,357982644067712,09/2020,72,YES,1,$13000,01/1999,2005,No


In [0]:
(df_cards.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_project.bronze.cards_data")
)

In [0]:
spark.sql("SELECT COUNT(*) FROM fraud_project.bronze.cards_data").show()

+--------+
|COUNT(*)|
+--------+
|    6146|
+--------+



In [0]:
az storage blob list --account-name stfrauddata001 --container-name bronze-landing --auth-mode login --output table

  File <command-8927038631493083>, line 1
    az storage blob list --account-name stfrauddata001 --container-name bronze-landing --auth-mode login --output table
       ^
SyntaxError: invalid syntax


In [0]:
storage_account_name = "stfrauddata001"  # your actual storage account name
storage_account_key = "sSPDqjuP4uI1Hcbx5h9gMrghfIHCTnA+eeYR3XhvbGpa4dKKg7mfSvuF8s/XySFS2euJLkWD4QbK+AStuiOInw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4683165113577092>, line 4
      1 storage_account_name = "stfrauddata001"  # your actual storage account name
      2 storage_account_key = "sSPDqjuP4uI1Hcbx5h9gMrghfIHCTnA+eeYR3XhvbGpa4dKKg7mfSvuF8s/XySFS2euJLkWD4QbK+AStuiOInw=="
----> 4 spark.conf.set(
      5     f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
      6     storage_account_key
      7 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:51, in RuntimeConf.set(self, key, value)
     49 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     50 operation = proto.ConfigRequest.Operation(set=op_set)
---> 51 result = self._client.config(operation)
     52 for warn in result.warnings:
     53     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/c

In [0]:
storage_account_name = "stfrauddata001"
storage_account_key = "sSPDqjuP4uI1Hcbx5h9gMrghfIHCTnA+eeYR3XhvbGpa4dKKg7mfSvuF8s/XySFS2euJLkWD4QbK+AStuiOInw=="

df_users = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
    .csv(f"abfss://bronze-landing@{storage_account_name}.dfs.core.windows.net/users_data.csv")
)
print(df_users.count())
display(df_users.limit(10))

2000


id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
(df_users.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_project.bronze.users_data")
)

In [0]:
spark.sql("SELECT COUNT(*) FROM fraud_project.bronze.users_data").show()
display(spark.sql("SELECT * FROM fraud_project.bronze.users_data LIMIT 10"))

+--------+
|COUNT(*)|
+--------+
|    2000|
+--------+



id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
storage_account_name = "stfrauddata001"  # confirm your actual name
storage_account_key = "sSPDqjuP4uI1Hcbx5h9gMrghfIHCTnA+eeYR3XhvbGpa4dKKg7mfSvuF8s/XySFS2euJLkWD4QbK+AStuiOInw=="

df_mcc = (spark.read
    .format("json")
    .option("multiLine", "true")
    .option("fs.azure.account.key." + storage_account_name + ".dfs.core.windows.net", storage_account_key)
    .load(f"abfss://bronze-landing@{storage_account_name}.dfs.core.windows.net/mcc_codes.json")
)
display(df_mcc)
df_mcc.printSchema()

1711,3000,3001,3005,3006,3007,3008,3009,3058,3066,3075,3132,3144,3174,3256,3260,3359,3387,3389,3390,3393,3395,3405,3504,3509,3596,3640,3684,3722,3730,3771,3775,3780,4111,4112,4121,4131,4214,4411,4511,4722,4784,4814,4829,4899,4900,5045,5094,5192,5193,5211,5251,5261,5300,5310,5311,5411,5499,5533,5541,5621,5651,5655,5661,5712,5719,5722,5732,5733,5812,5813,5814,5815,5816,5912,5921,5932,5941,5942,5947,5970,5977,6300,7011,7210,7230,7276,7349,7393,7531,7538,7542,7549,7801,7802,7832,7922,7995,7996,8011,8021,8041,8043,8049,8062,8099,8111,8931,9402
"Heating, Plumbing, Air Conditioning Contractors",Steelworks,Steel Products Manufacturing,Miscellaneous Metal Fabrication,Miscellaneous Fabricated Metal Products,Coated and Laminated Products,Steel Drums and Barrels,Fabricated Structural Metal Products,"Tools, Parts, Supplies Manufacturing",Miscellaneous Metals,"Bolt, Nut, Screw, Rivet Manufacturing",Leather Goods,Floor Covering Stores,Upholstery and Drapery Stores,"Brick, Stone, and Related Materials",Pottery and Ceramics,Non-Ferrous Metal Foundries,"Electroplating, Plating, Polishing Services",Non-Precious Metal Services,Miscellaneous Metalwork,Heat Treating Metal Services,Welding Repair,Ironwork,Gardening Supplies,Industrial Equipment and Supplies,Miscellaneous Machinery and Parts Manufacturing,"Lighting, Fixtures, Electrical Supplies",Semiconductors and Related Devices,Passenger Railways,Ship Chandlers,Railroad Passenger Transport,Railroad Freight,Computer Network Services,Local and Suburban Commuter Transportation,Passenger Railways,Taxicabs and Limousines,Bus Lines,Motor Freight Carriers and Trucking,Cruise Lines,Airlines,Travel Agencies,Tolls and Bridge Fees,Telecommunication Services,Money Transfer,"Cable, Satellite, and Other Pay Television Services","Utilities - Electric, Gas, Water, Sanitary","Computers, Computer Peripheral Equipment",Precious Stones and Metals,"Books, Periodicals, Newspapers","Florists Supplies, Nursery Stock and Flowers",Lumber and Building Materials,Hardware Stores,Lawn and Garden Supply Stores,Wholesale Clubs,Discount Stores,Department Stores,"Grocery Stores, Supermarkets",Miscellaneous Food Stores,Automotive Parts and Accessories Stores,Service Stations,Women's Ready-To-Wear Stores,Family Clothing Stores,"Sports Apparel, Riding Apparel Stores",Shoe Stores,"Furniture, Home Furnishings, and Equipment Stores",Miscellaneous Home Furnishing Stores,Household Appliance Stores,Electronics Stores,Music Stores - Musical Instruments,Eating Places and Restaurants,Drinking Places (Alcoholic Beverages),Fast Food Restaurants,"Digital Goods - Media, Books, Apps",Digital Goods - Games,Drug Stores and Pharmacies,"Package Stores, Beer, Wine, Liquor",Antique Shops,Sporting Goods Stores,Book Stores,"Gift, Card, Novelty Stores","Artist Supply Stores, Craft Shops",Cosmetic Stores,"Insurance Sales, Underwriting","Lodging - Hotels, Motels, Resorts",Laundry Services,Beauty and Barber Shops,Tax Preparation Services,Cleaning and Maintenance Services,"Detective Agencies, Security Services",Automotive Body Repair Shops,Automotive Service Shops,Car Washes,Towing Services,"Athletic Fields, Commercial Sports","Recreational Sports, Clubs",Motion Picture Theaters,Theatrical Producers,"Betting (including Lottery Tickets, Casinos)","Amusement Parks, Carnivals, Circuses","Doctors, Physicians",Dentists and Orthodontists,Chiropractors,"Optometrists, Optical Goods and Eyeglasses",Podiatrists,Hospitals,Medical Services,Legal Services and Attorneys,"Accounting, Auditing, and Bookkeeping Services",Postal Services - Government Only


root
 |-- 1711: string (nullable = true)
 |-- 3000: string (nullable = true)
 |-- 3001: string (nullable = true)
 |-- 3005: string (nullable = true)
 |-- 3006: string (nullable = true)
 |-- 3007: string (nullable = true)
 |-- 3008: string (nullable = true)
 |-- 3009: string (nullable = true)
 |-- 3058: string (nullable = true)
 |-- 3066: string (nullable = true)
 |-- 3075: string (nullable = true)
 |-- 3132: string (nullable = true)
 |-- 3144: string (nullable = true)
 |-- 3174: string (nullable = true)
 |-- 3256: string (nullable = true)
 |-- 3260: string (nullable = true)
 |-- 3359: string (nullable = true)
 |-- 3387: string (nullable = true)
 |-- 3389: string (nullable = true)
 |-- 3390: string (nullable = true)
 |-- 3393: string (nullable = true)
 |-- 3395: string (nullable = true)
 |-- 3405: string (nullable = true)
 |-- 3504: string (nullable = true)
 |-- 3509: string (nullable = true)
 |-- 3596: string (nullable = true)
 |-- 3640: string (nullable = true)
 |-- 3684: string (null

In [0]:
storage_account_name = "stfrauddata001"
storage_account_key = "sSPDqjuP4uI1Hcbx5h9gMrghfIHCTnA+eeYR3XhvbGpa4dKKg7mfSvuF8s/XySFS2euJLkWD4QbK+AStuiOInw=="

In [0]:
# Read the wide multi-line JSON (1 row, ~109 columns)
df_mcc_wide = (spark.read
    .option("fs.azure.account.key." + storage_account_name + ".dfs.core.windows.net", storage_account_key)
    .option("multiLine", "true")
    .json(f"abfss://bronze-landing@{storage_account_name}.dfs.core.windows.net/mcc_codes.json")
)

# Pivot the single wide row into a proper two-column table
wide_row = df_mcc_wide.collect()[0].asDict()

mcc_df = spark.createDataFrame(
    [(k, v) for k, v in wide_row.items()],
    ["mcc_code", "mcc_description"]
)

print("Row count:", mcc_df.count())
display(mcc_df)

Row count: 109


mcc_code,mcc_description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
mcc_df.write.format("delta").mode("overwrite").saveAsTable("fraud_project.bronze.mcc_codes")

spark.sql("SELECT COUNT(*) FROM fraud_project.bronze.mcc_codes").show()

+--------+
|COUNT(*)|
+--------+
|     109|
+--------+



In [0]:
# Read as text to avoid gRPC message size limit with wide schemas
raw_labels = (spark.read
    .option("fs.azure.account.key." + storage_account_name + ".dfs.core.windows.net", storage_account_key)
    .text(f"abfss://bronze-landing@{storage_account_name}.dfs.core.windows.net/train_fraud_labels.json")
)

# Parse JSON manually to handle very wide schema
import json

json_text = raw_labels.collect()[0][0]
labels_dict = json.loads(json_text)

# Extract nested "target" dictionary
target_labels = labels_dict["target"]

# Convert to two-column format: transaction_id, is_fraud
df_labels = spark.createDataFrame(
    [(k, v) for k, v in target_labels.items()],
    ["transaction_id", "is_fraud"]
)

print(f"Number of fraud labels: {df_labels.count()}")
display(df_labels.limit(10))

Number of fraud labels: 8914963


transaction_id,is_fraud
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
import json

raw_text_df = (spark.read
    .format("text")
    .option("wholetext", "true")
    .option("fs.azure.account.key." + storage_account_name + ".dfs.core.windows.net", storage_account_key)
    .load(f"abfss://bronze-landing@{storage_account_name}.dfs.core.windows.net/train_fraud_labels.json")
)

raw_json_string = raw_text_df.collect()[0]["value"]
print("Raw text length (chars):", len(raw_json_string))

Raw text length (chars): 159083088


In [0]:
import json

labels_dict = json.loads(raw_json_string)
print(type(labels_dict))

if isinstance(labels_dict, dict):
    print("Top-level keys:", list(labels_dict.keys())[:5])

<class 'dict'>
Top-level keys: ['target']


In [0]:
inner = labels_dict.get("target", labels_dict) if isinstance(labels_dict, dict) else labels_dict

print("Number of entries:", len(inner))
sample_items = list(inner.items())[:5]
print(sample_items)

Number of entries: 8914963
[('10649266', 'No'), ('23410063', 'No'), ('9316588', 'No'), ('12478022', 'No'), ('9558530', 'No')]


In [0]:
labels_df = spark.createDataFrame(
    [(k, v) for k, v in inner.items()],
    ["transaction_id", "is_fraud_label"]
)

print("Row count:", labels_df.count())
display(labels_df.limit(10))

Row count: 8914963


transaction_id,is_fraud_label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
labels_df.write.format("delta").mode("overwrite").saveAsTable("fraud_project.bronze.train_fraud_labels")

In [0]:
spark.sql("SELECT COUNT(*) FROM fraud_project.bronze.train_fraud_labels").show()

+--------+
|COUNT(*)|
+--------+
| 8914963|
+--------+



In [0]:
tables = ["transactions_data", "cards_data", "users_data", "mcc_codes", "train_fraud_labels"]
for t in tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM fraud_project.bronze.{t}").collect()[0]["cnt"]
    print(f"{t}: {cnt} rows")

transactions_data: 13305915 rows
cards_data: 6146 rows
users_data: 2000 rows
mcc_codes: 109 rows
train_fraud_labels: 8914963 rows
